# EPFL Hackathon 2026 - Data Preprocesing

# 📊 Synthetic Underwriting Dataset – Fully Structured Mathematical Description

This dataset simulates an insurance underwriting triage setting with:

- Individual fraud risk modeling
- Economic valuation of manual review
- Latent network fraud structure
- Pairwise interaction effects for QUBO optimization

The construction is mathematically and economically consistent.

---

# 1️⃣ Optimization Objective

We model the decision problem:

$$
\max_{x \in \{0,1\}^N}
\left(
\sum_i v_i x_i
+
\sum_{i<j} w_{ij} x_i x_j
\right)
\quad
\text{s.t. } \sum_i x_i = k
$$

Where:

- $v_i$ = expected value of reviewing case $i$
- $w_{ij}$ = additional joint value of reviewing $i$ and $j$ together
- $k$ = review capacity

---

# 2️⃣ Latent World Structure

## 2.1 Entity-Level Latent Risk

Each broker, garage, and device has a latent risk parameter:

$$
\theta^{broker}_b,\quad
\theta^{garage}_g,\quad
\theta^{device}_d
$$

These are sampled from:

$$
\theta \sim \mathcal{N}(0, \sigma^2)
$$

They represent persistent structural ecosystem risk.

---

## 2.2 Fraud Rings (Latent Network Structure)

A fraction of cases belong to hidden fraud rings.

Each ring reuses:

- Brokers
- Garages
- Devices

This induces structural correlation:

$$
P(\text{fraud}_j \mid \text{fraud}_i, \text{shared entity}) 
>
P(\text{fraud})
$$

---

# 3️⃣ Case-Level Feature Generation

Each case $i$ includes:

- `claim_type`
- `claim_amount`
- `policy_age_days`
- `prior_claims`
- `complexity`
- `review_minutes`
- `broker_id`
- `garage_id`
- `device_id`
- `region_id`

Claim amounts follow type-dependent log-normal distributions:

$$
\text{claim\_amount} \sim \text{LogNormal}(\mu_{\text{type}}, \sigma)
$$

---

# 4️⃣ Node-Level Network Risk Score

For each case:

$$
\text{entity\_risk\_score}_i =
0.30\,\theta^{broker}_{b(i)}
+
0.45\,\theta^{garage}_{g(i)}
+
0.25\,\theta^{device}_{d(i)}
$$

This score reflects structural exposure from associated entities.

---

# 5️⃣ Pre-Review Fraud Probability

Fraud probability is modeled via logistic regression:

$$
\text{score}_i =
\beta_0
+ \beta_1 \log(\text{claim\_amount})
+ \beta_2 \mathbf{1}(\text{young policy})
+ \beta_3 \text{prior claims}
+ \beta_4 \text{entity\_risk\_score}_i
+ \beta_5 \mathbf{1}(\text{in ring})
+ \beta_6 \text{type shift}
$$

Then:

$$
p_i = \sigma(\text{score}_i)
= \frac{1}{1 + e^{-\text{score}_i}}
$$

This represents the automated risk estimate before manual review.

---

# 6️⃣ Economic Translation

## 6.1 Preventable Loss

Leakage depends on claim type:

$$
L_i = \lambda_{\text{type}} \cdot \text{claim\_amount}_i
$$

Where:

- Auto → $\lambda = 0.30$
- Home → $\lambda = 0.40$
- Health → $\lambda = 0.20$

---

## 6.2 Review Cost

$$
c_i =
\frac{\text{review\_minutes}_i}{60}
\cdot
\text{hourly\_cost}
$$

---

## 6.3 Expected Value of Review

$$
v_i =
\alpha \, p_i \, L_i
-
c_i
$$

Where:

- $\alpha$ = detection effectiveness
- $p_i$ = fraud probability
- $L_i$ = preventable exposure
- $c_i$ = operational cost

This is the linear component of the QUBO.

---

# 7️⃣ Pairwise Interaction Term $w_{ij}$

The correlation term represents incremental economic value of joint investigation.

---

## 7.1 Link Strength

Shared-entity indicators define:

$$
s_{ij}
=
1 -
\prod_e
\left(
1 - \rho_e \mathbf{1}[\text{match}_e(i,j)]
\right)
$$

Where:

- $\rho_e$ = strength of each shared entity signal
- $s_{ij} \in [0,1]$

---

## 7.2 Economic Interaction

$$
w_{ij}
=
s_{ij}
\left(
\kappa \sqrt{v_i v_j}
+
\mu \min(c_i, c_j)
+
\pi \cdot \text{rarity bonus}
\right)
-
\delta \cdot \text{redundancy penalty}
$$

Interpretation:

- $\sqrt{v_i v_j}$ → large joint economic exposure
- $\min(c_i,c_j)$ → investigation cost sharing
- rarity bonus → rare shared entity strengthens evidence
- redundancy penalty → penalizes trivial duplicates

---

# 8️⃣ Conceptual Summary

This dataset provides:

- Individual risk modeling
- Economic valuation
- Structured fraud network generation
- Interaction terms with economic meaning
- Clean QUBO / Ising mapping

---

# 🔟 Key Distinction

The interaction term is NOT a statistical correlation:

$$
\text{It is an incremental economic value term}
$$

It models how joint investigation changes expected savings.

In [2]:
import numpy as np
import pandas as pd

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def generate_underwriting_dataset_consistent(
    N=24,
    n_brokers=10,
    n_garages=12,
    n_regions=6,
    n_devices=40,
    n_rings=3,
    ring_fraction=0.35,
    seed=0,
):
    """
    Fully consistent synthetic dataset:
      - Latent entity risks (broker/garage/device) are sampled first
      - Cases sample entities (rings reuse entities)
      - p_fraud is computed from features + latent entity risks (no leakage)
      - severity_L uses leakage_by_type * claim_amount
      - v_i = alpha * p_fraud * severity_L - review_cost
      - w_ij is built from (shared entity links) * (economic uplift + cost sharing + evidence rarity) - redundancy
    """
    rng = np.random.default_rng(seed)

    # -----------------------
    # 1) Global parameters
    # -----------------------
    claim_types = np.array(["auto", "home", "health"])
    type_probs  = np.array([0.55, 0.30, 0.15])

    leakage_by_type = {"auto": 0.30, "home": 0.40, "health": 0.20}

    alpha_effectiveness = 0.55   # P(review catches/mitigates | problematic)
    hourly_cost = 60.0           # cost per hour of reviewer

    # Link strength weights (how strong each shared-entity signal is)
    rho = {
        "broker": 0.25,
        "garage": 0.50,
        "device": 0.60,
        "region_type": 0.15
    }

    # Synergy scaling
    kappa_uplift = 0.18   # scales "network uplift" in money
    mu_costshare = 0.50   # fraction of min(review_cost) you save by bundling
    pi_evidence  = 30.0   # evidence/prosecution bonus scale (rarer shared entities => larger)
    delta_redund = 0.25   # redundancy penalty scale

    # -----------------------
    # 2) Latent entity risks (no circularity)
    # -----------------------
    # These are fixed unknown propensities in the world (what the insurer has learned historically)
    broker_risk_latent = rng.normal(0.0, 0.35, size=n_brokers)
    garage_risk_latent = rng.normal(0.0, 0.45, size=n_garages)
    device_risk_latent = rng.normal(0.0, 0.30, size=n_devices)

    # -----------------------
    # 3) Fraud rings (latent structure that reuses entities)
    # -----------------------
    in_ring = rng.random(N) < ring_fraction
    ring_id_latent = np.zeros(N, dtype=int)
    ring_id_latent[in_ring] = rng.integers(1, n_rings + 1, size=in_ring.sum())

    # Each ring uses small pools of entities (creates correlation)
    def make_pool(n_total, frac):
        m = max(1, int(np.round(n_total * frac)))
        return rng.choice(n_total, size=m, replace=False)

    ring_broker_pool = {r: make_pool(n_brokers, 0.25) for r in range(1, n_rings+1)}
    ring_garage_pool = {r: make_pool(n_garages, 0.25) for r in range(1, n_rings+1)}
    ring_device_pool = {r: make_pool(n_devices, 0.30) for r in range(1, n_rings+1)}

    def pick_entity(ring, pool_dict, n_total, reuse_prob=0.80):
        if ring == 0:
            return int(rng.integers(0, n_total))
        # ring tends to reuse within pool
        if rng.random() < reuse_prob:
            return int(rng.choice(pool_dict[ring]))
        return int(rng.integers(0, n_total))

    # -----------------------
    # 4) Generate case-level features
    # -----------------------
    rows = []
    for i in range(N):
        ctype = str(rng.choice(claim_types, p=type_probs))
        region = int(rng.integers(0, n_regions))
        ring = int(ring_id_latent[i])

        # Claim amount: type-dependent lognormal (right-skewed)
        if ctype == "auto":
            amount = float(rng.lognormal(mean=8.2, sigma=0.55))   # ~ few k to ~20k
        elif ctype == "home":
            amount = float(rng.lognormal(mean=8.7, sigma=0.60))   # larger typical
        else:
            amount = float(rng.lognormal(mean=7.9, sigma=0.50))   # smaller typical

        policy_age_days = int(rng.integers(5, 3650))
        prior_claims = int(rng.poisson(0.6))

        complexity = float(np.clip(rng.normal(1.0, 0.35), 0.3, 2.0))
        review_minutes = int(np.clip(20 + 35 * complexity + 8 * prior_claims, 15, 120))

        broker = pick_entity(ring, ring_broker_pool, n_brokers)
        garage = pick_entity(ring, ring_garage_pool, n_garages)
        device = pick_entity(ring, ring_device_pool, n_devices)

        # Node-level network exposure score (from latent entity risks)
        # (No circularity: uses broker_risk_latent etc.)
        entity_risk_score = (
            0.30 * broker_risk_latent[broker] +
            0.45 * garage_risk_latent[garage] +
            0.25 * device_risk_latent[device]
        )

        # -----------------------
        # 5) Pre-review fraud probability p_fraud
        # -----------------------
        # Uses only "metadata" + latent entity risks + mild ring effect (rings are harder)
        # This is what your model would estimate automatically.
        log_amount = np.log(max(amount, 1.0))
        young_policy = 1.0 if policy_age_days < 60 else 0.0

        type_shift = {"auto": 0.15, "home": 0.05, "health": -0.05}[ctype]

        score = (
            -3.2
            + type_shift
            + 0.50 * (log_amount - 8.2)
            + 0.65 * young_policy
            + 0.35 * prior_claims
            + 0.85 * entity_risk_score
            + 0.55 * (1.0 if ring != 0 else 0.0)
        )
        p_fraud = float(sigmoid(score))

        rows.append({
            "case_id": i,
            "claim_type": ctype,
            "region_id": region,
            "claim_amount": amount,
            "policy_age_days": policy_age_days,
            "prior_claims": prior_claims,
            "complexity": complexity,
            "review_minutes": review_minutes,
            "broker_id": broker,
            "garage_id": garage,
            "device_id": device,
            "entity_risk_score": float(entity_risk_score),
            "p_fraud": p_fraud,
            "ring_id_latent": ring,
        })

    df = pd.DataFrame(rows)

    # -----------------------
    # 6) Economics: severity_L, review_cost, v_i
    # -----------------------
    df["severity_L"] = df.apply(
        lambda r: leakage_by_type[r["claim_type"]] * r["claim_amount"],
        axis=1
    )
    df["review_cost"] = (df["review_minutes"] / 60.0) * hourly_cost

    # Expected value of reviewing:
    # v_i = alpha * p_fraud * severity_L - review_cost
    df["v_i"] = alpha_effectiveness * df["p_fraud"] * df["severity_L"] - df["review_cost"]

    # -----------------------
    # 7) Pairwise synergy matrix W (w_ij)
    # -----------------------
    # Define:
    #   s_ij = combined link strength from shared entities (in [0,1])
    #   evidence bonus higher if shared entity is rare in this batch
    #   redundancy penalty if cases are "too similar" and low incremental info
    N = len(df)
    W = np.zeros((N, N), dtype=float)

    # rarity counts within the batch (simple proxy)
    broker_counts = df["broker_id"].value_counts().to_dict()
    garage_counts = df["garage_id"].value_counts().to_dict()
    device_counts = df["device_id"].value_counts().to_dict()

    def rarity_bonus(count):
        # fewer occurrences => higher bonus, bounded
        return float(np.clip(1.0 / max(count, 1), 0.02, 0.50))

    v = df["v_i"].to_numpy()
    c = df["review_cost"].to_numpy()

    for i in range(N):
        for j in range(i+1, N):
            same_broker = df.loc[i, "broker_id"] == df.loc[j, "broker_id"]
            same_garage = df.loc[i, "garage_id"] == df.loc[j, "garage_id"]
            same_device = df.loc[i, "device_id"] == df.loc[j, "device_id"]
            same_region_type = (df.loc[i, "region_id"] == df.loc[j, "region_id"]) and (df.loc[i, "claim_type"] == df.loc[j, "claim_type"])

            # Combined link strength in [0,1] (no correlation coefficient!)
            s = 1.0
            if same_broker:      s *= (1.0 - rho["broker"])
            if same_garage:      s *= (1.0 - rho["garage"])
            if same_device:      s *= (1.0 - rho["device"])
            if same_region_type: s *= (1.0 - rho["region_type"])
            s = 1.0 - s

            if s == 0.0:
                continue

            # Evidence bonus: rare shared entities are stronger "proof" of a ring
            ev = 0.0
            if same_broker:
                ev += rarity_bonus(broker_counts[int(df.loc[i, "broker_id"])])
            if same_garage:
                ev += 1.5 * rarity_bonus(garage_counts[int(df.loc[i, "garage_id"])])
            if same_device:
                ev += 2.0 * rarity_bonus(device_counts[int(df.loc[i, "device_id"])])

            # Uplift term: only if both are economically promising
            vi = max(v[i], 0.0)
            vj = max(v[j], 0.0)
            uplift = kappa_uplift * np.sqrt(vi * vj)

            # Cost-sharing: handling linked cases together saves time/cost
            costshare = mu_costshare * min(c[i], c[j])

            # Redundancy penalty: if cases are extremely similar & low value, second adds little info
            # (you can tweak this rule)
            redundancy = 1.0 if (same_region_type and same_broker and abs(v[i]-v[j]) < 5.0 and max(vi, vj) < 20.0) else 0.0
            redund_pen = delta_redund * redundancy * (abs(v[i]) + abs(v[j])) * 0.10

            wij = s * (uplift + costshare + pi_evidence * ev) - redund_pen
            W[i, j] = wij
            W[j, i] = wij

    # Optional sparsification: keep only strongest edges for realism & easier optimization
    edges = [(i, j, W[i, j]) for i in range(N) for j in range(i+1, N) if W[i, j] != 0.0]
    edges = sorted(edges, key=lambda x: abs(x[2]), reverse=True)
    keep_m = max(1, 2 * N)  # keep ~2N edges
    keep = set((i, j) for i, j, _ in edges[:keep_m])
    for i, j, _ in edges:
        if (i, j) not in keep:
            W[i, j] = 0.0
            W[j, i] = 0.0

    return df, W

if __name__ == "__main__":
    df, W = generate_underwriting_dataset_consistent(N=50, seed=42)
    print(df.head(8)[[
        "case_id","claim_type","claim_amount","policy_age_days","prior_claims",
        "broker_id","garage_id","device_id","entity_risk_score","p_fraud",
        "severity_L","review_cost","v_i","ring_id_latent"
    ]])
    print("Non-zero edges in W:", int((np.abs(W) > 1e-12).sum() // 2))

   case_id claim_type  claim_amount  policy_age_days  prior_claims  broker_id  \
0        0       auto   3560.230648             3618             0          9   
1        1     health   5940.482442             3255             2          0   
2        2       auto   4609.229757             2689             0          7   
3        3       auto   3298.636343             3034             0          0   
4        4       auto   3327.932122             3410             0          0   
5        5       home   4912.439475              733             1          4   
6        6       auto   3040.507734              486             2          1   
7        7       auto  14539.301654             1532             2          6   

   garage_id  device_id  entity_risk_score   p_fraud   severity_L  \
0          3         17           0.187491  0.052063  1068.069195   
1          9         13           0.106558  0.098442  1188.096488   
2         11          9          -0.201575  0.042966  1382.7689

In [3]:
import numpy as np
import pandas as pd

from qiskit.circuit.library import QAOAAnsatz
from qiskit_aer import Aer
from qiskit_aer.primitives import Estimator 
from qiskit import transpile
from scipy.optimize import minimize

In [4]:
# 1) Load dataset
df = pd.read_csv("underwriting_dataset.csv")
df.columns = df.columns.str.strip()

# Use "v_i" directly (expected value of investigating)
v = df["v_i"].astype(float).to_numpy()
n = len(df)

# Choose exactly K to investigate (adjust as you want)
K = min(8, n)  # investigate 8 out of 24
lam = 1.0      # network synergy weight
rho = 200.0    # penalty weight for enforcing sum x_i = K

In [115]:
# 3) Build QUBO  

qp = QuadraticProgram("fraud_alert_prioritization")

for i in range(n):
    qp.binary_var(f"x_{i}")

linear = {}
quadratic = {}

def add_q(u, v, c):
    key = tuple(sorted((u, v)))
    quadratic[key] = quadratic.get(key, 0.0) + float(c)

# (A) -sum v_i x_i
for i in range(n):
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) - float(v[i])

# (B) -lam * sum W_ij x_i x_j
for i in range(n):
    for j in range(i + 1, n):
        if W[i, j] != 0:
            add_q(f"x_{i}", f"x_{j}", -lam * float(W[i, j]))

# (C) EXACTLY-K constraint via penalty: rho*(sum_i x_i - K)^2
# Expand: rho*(sum x)^2 - 2*rho*K*(sum x) + const
# and (sum x)^2 = sum x_i + 2*sum_{i<j} x_i x_j because x_i^2 = x_i

for i in range(n):
    # from rho*(sum x)^2 -> +rho*x_i
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) + rho
    # from -2*rho*K*(sum x) -> -2*rho*K*x_i
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) - 2.0 * rho * K

for i in range(n):
    for j in range(i + 1, n):
        # from rho*(sum x)^2 -> +2*rho*x_i*x_j
        add_q(f"x_{i}", f"x_{j}", 2.0 * rho)

qp.minimize(linear=linear, quadratic=quadratic)
qubo = QuadraticProgramToQubo().convert(qp)

In [116]:
# 4) Convert QUBO -> Ising
H, offset = qubo.to_ising()

reps = 2
ansatz = QAOAAnsatz(cost_operator=H, reps=reps)

estimator = Estimator()

def energy(theta):
    job = estimator.run([ansatz], [H], [theta])
    return float(job.result().values[0])

theta0 = np.random.default_rng(0).uniform(0, 2*np.pi, size=ansatz.num_parameters)
res_opt = minimize(energy, theta0, method="COBYLA", options={"maxiter": 200})
theta_star = res_opt.x

qc = ansatz.assign_parameters(theta_star)
qc.measure_all()

backend = Aer.get_backend("aer_simulator")
tqc = transpile(qc, backend)
counts = backend.run(tqc, shots=3000).result().get_counts()

best = max(counts, key=counts.get)

bits = np.array([int(b) for b in best[::-1]], dtype=int)

# Since the corrected QUBO has ONLY x_0..x_{n-1}, keep first n bits
x = bits[:n]
selected_idx = np.where(x == 1)[0]

In [118]:
# 5) Report 

value = float(np.sum(v * x))

synergy = 0.0
for i in range(n):
    for j in range(i + 1, n):
        synergy += W[i, j] * x[i] * x[j]
synergy = float(synergy)

print("Most frequent bitstring:", best)
print("Selected cases:", selected_idx.tolist())
print("Selected count:", int(x.sum()), "(target K =", K, ")")
print("Sum individual value:", value)
print("Network synergy term:", lam * synergy)
print("Total (value + synergy):", value + lam * synergy)

display(df.loc[selected_idx].reset_index(drop=True))

Most frequent bitstring: 101001000101000011111000
Selected cases: [3, 4, 5, 6, 7, 12, 14, 18, 21, 23]
Selected count: 10 (target K = 8 )
Sum individual value: 2474.7764008383
Network synergy term: 34.89999999999999
Total (value + synergy): 2509.6764008383


,case_id,claim_type,region_id,claim_amount,policy_age_days,prior_claims,broker_id,garage_id,device_id,review_minutes,p_fraud,ring_id_latent,severity_L,review_cost,v_i
0,3,auto,0,4160.428011,545,0,5,1,17,56,0.284220,1,1248.128403,56.0,139.108453
1,4,auto,4,1974.703798,1914,1,5,8,9,68,0.360451,1,592.411139,68.0,49.444431
2,5,auto,4,1864.767982,187,1,7,3,15,77,0.357810,3,559.430395,77.0,33.093401
3,6,health,3,3393.757134,1039,0,4,0,20,56,0.043008,0,678.751427,56.0,-39.944485
4,7,home,4,4920.212978,1618,0,3,3,25,53,0.273989,2,1968.085191,53.0,243.578013
5,12,auto,4,3759.536216,2598,0,0,3,25,52,0.290947,2,1127.860865,52.0,128.481259
6,14,auto,3,20918.088114,1168,1,4,8,4,74,0.561679,2,6275.426434,74.0,1864.625098
7,18,health,1,3021.150575,706,0,2,3,21,89,0.206564,3,604.230115,89.0,-20.353175
8,21,health,3,1092.420953,2667,0,4,3,1,66,0.138480,1,218.484191,66.0,-49.359406
9,23,auto,0,2957.815911,3610,1,7,1,2,58,0.377229,3,887.344773,58.0,126.102812
